# Pipeline v12 — Evolución producto-período multisemilla

Esta versión cambia la unidad de modelado a **<producto, período>**. La información de clientes
se conserva únicamente como métricas agregadas por producto-mes. Compara un LightGBM global,
especialistas por curvas de producto y un baseline estacional. La selección se realiza con
validación temporal usando las tres semillas: `109903`, `109927` y `203999`.


## 0. Diseño experimental

| Modelo | Unidad | Semillas | Papel |
|---|---|---|---|
| Global | producto-período | 3 | relaciones comunes entre productos |
| Especialistas | producto-período por cluster de curva | 3 | comportamientos distintos de demanda |
| Estacional | producto-período | determinista | febrero comparable del año anterior |

Los pesos y una calibración pequeña se eligen con validación. El test temporal queda separado.
Sólo se genera un submission final y el envío a Kaggle está desactivado por defecto.


In [1]:
# Instalación e imports
%pip install -q lightgbm pandas numpy scikit-learn


Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path
import json, os, shutil, subprocess, time
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

SEEDS = [109903, 109927, 203999]
HORIZONTE = 2
ORIGEN_INFERENCIA = 201912
VALID_ORIGINS = [201812, 201906]
TEST_ORIGINS = [201910]
K_CLUSTERS = 6
N_THREADS = max(1, (os.cpu_count() or 2) - 1)
SUBMIT_V12 = False

def resolver_bucket():
    for p in [Path.home()/"buckets"/"b1", Path("/home/alojarse_en_baires/buckets/b1"), Path("/content/drive/MyDrive/labo3")]:
        if p.exists(): return p
    return Path.cwd()

BUCKET = resolver_bucket()
RAW = BUCKET / "datasets"
OUT = BUCKET / "pipeline_v12_producto_periodo"
OUT.mkdir(parents=True, exist_ok=True)
print("BUCKET:", BUCKET)
print("Semillas:", SEEDS)
print("Validación:", VALID_ORIGINS, "Test:", TEST_ORIGINS, "Inferencia:", ORIGEN_INFERENCIA)


BUCKET: /home/ds/buckets/b1
Semillas: [109903, 109927, 203999]
Validación: [201812, 201906] Test: [201910] Inferencia: 201912


## 1. Datos crudos y universo

La venta se agrega primero por cliente-producto-mes para calcular concentración y retención,
y después queda una única fila por producto-mes para modelar.


In [3]:
def descargar(nombre):
    destino = RAW / nombre
    if destino.exists(): return
    RAW.mkdir(parents=True, exist_ok=True)
    url = f"https://storage.googleapis.com/open-courses/austral2026-5da5/labo3/{nombre}"
    subprocess.run(["wget", url, "-O", str(destino)], check=True)

for nombre in ["sell-in.txt.gz", "tb_productos.txt", "product_id_apredecir201912.txt"]:
    descargar(nombre)

sell = pd.read_csv(RAW/"sell-in.txt.gz", sep="\t")
productos = pd.read_csv(RAW/"tb_productos.txt", sep="\t")
oficial = pd.read_csv(RAW/"product_id_apredecir201912.txt", sep="\t", usecols=["product_id"])
for d in [sell, productos, oficial]:
    d["product_id"] = pd.to_numeric(d["product_id"], errors="raise").astype("int64")
sell["customer_id"] = pd.to_numeric(sell["customer_id"], errors="raise").astype("int64")
sell["periodo"] = pd.to_numeric(sell["periodo"], errors="raise").astype("int64")
sell["tn"] = pd.to_numeric(sell["tn"], errors="coerce").fillna(0.0)
print("sell-in:", sell.shape, "productos objetivo:", oficial.product_id.nunique())


sell-in: (2945818, 7) productos objetivo: 780


## 2. Calendario y métricas agregadas de clientes

Las variables de clientes no identifican al cliente en el modelo. Resumen cuántos compraron,
cuántos repitieron y cuán concentrada estuvo la venta de cada producto en cada período.


In [4]:
def add_months_scalar(p, h):
    y, m = divmod(int(p), 100)
    z = y*12 + (m-1) + h
    return (z//12)*100 + (z%12) + 1

def month_index(s):
    s = pd.Series(s).astype(int)
    return (s//100)*12 + (s%100)-1

cp = sell.groupby(["customer_id","product_id","periodo"], as_index=False)["tn"].sum()
cp = cp[cp.tn > 0].copy()

activos = cp.groupby(["product_id","periodo"]).customer_id.nunique().rename("clientes_activos").reset_index()
prev = cp[["customer_id","product_id","periodo"]].copy()
prev["periodo"] = prev["periodo"].map(lambda p: add_months_scalar(p, 1))
prev["compro_anterior"] = 1
ret = cp.merge(prev, on=["customer_id","product_id","periodo"], how="left")
ret["retenido"] = ret.compro_anterior.fillna(0).astype(int)
retenidos = ret.groupby(["product_id","periodo"]).retenido.sum().rename("clientes_retenidos").reset_index()

def concentracion(x):
    a = np.asarray(x, dtype=float)
    total = a.sum()
    if total <= 0: return pd.Series({"hhi_clientes":0.0,"share_top1":0.0,"share_top5":0.0})
    sh = np.sort(a/total)[::-1]
    return pd.Series({"hhi_clientes":float(np.square(sh).sum()),"share_top1":float(sh[0]),"share_top5":float(sh[:5].sum())})

conc = cp.groupby(["product_id","periodo"])["tn"].apply(concentracion).unstack().reset_index()
metricas_cli = activos.merge(retenidos, on=["product_id","periodo"], how="left").merge(conc, on=["product_id","periodo"], how="left")
metricas_cli["clientes_retenidos"] = metricas_cli.clientes_retenidos.fillna(0)
print("métricas cliente agregadas:", metricas_cli.shape)


métricas cliente agregadas: (31243, 7)


## 3. Panel denso producto-período

La grilla es pequeña: productos × meses. Se preservan los ceros sin replicarlos por cliente.


In [5]:
agg = sell.groupby(["product_id","periodo"], as_index=False).tn.sum()
periodos = sorted(sell.periodo.unique())
universo = sorted(sell.product_id.unique())
grid = pd.MultiIndex.from_product([universo, periodos], names=["product_id","periodo"]).to_frame(index=False)
panel = grid.merge(agg, on=["product_id","periodo"], how="left")
panel["tn"] = panel.tn.fillna(0.0).clip(lower=0)
panel = panel.merge(metricas_cli, on=["product_id","periodo"], how="left")
for c in ["clientes_activos","clientes_retenidos","hhi_clientes","share_top1","share_top5"]:
    panel[c] = panel[c].fillna(0.0)
meta_cols = [c for c in ["product_id","cat3","brand"] if c in productos.columns]
panel = panel.merge(productos[meta_cols].drop_duplicates("product_id"), on="product_id", how="left")
panel["cat3"] = panel.get("cat3", "SIN_CAT").fillna("SIN_CAT").astype(str)
panel["brand"] = panel.get("brand", "SIN_MARCA").fillna("SIN_MARCA").astype(str)
panel = panel.sort_values(["product_id","periodo"]).reset_index(drop=True)
print("panel producto-período:", panel.shape, "ceros:", int((panel.tn==0).sum()))


panel producto-período: (44388, 10) ceros: 13145


## 4. Clusters causales de curvas de producto

Los clusters se ajustan sólo con ventas hasta octubre de 2018, antes de los orígenes de
validación. Cada curva se normaliza por su media para agrupar forma, no tamaño.


In [6]:
CORTE_CLUSTER = 201810
curvas = panel[panel.periodo <= CORTE_CLUSTER].pivot(index="product_id", columns="periodo", values="tn").fillna(0.0)
media = curvas.mean(axis=1).replace(0, 1.0)
X_curvas = curvas.div(media, axis=0)
kmeans = KMeans(n_clusters=K_CLUSTERS, random_state=SEEDS[0], n_init=20)
labels = kmeans.fit_predict(X_curvas)
map_cluster = pd.DataFrame({"product_id":X_curvas.index.astype(int), "cluster_id":labels.astype(int)})
panel = panel.merge(map_cluster, on="product_id", how="left")
panel["cluster_id"] = panel.cluster_id.fillna(0).astype(int)
print(map_cluster.cluster_id.value_counts().sort_index())
print("silhouette:", round(silhouette_score(X_curvas, labels), 4))


cluster_id
0    194
1     32
2    953
3     28
4      9
5     17
Name: count, dtype: int64
silhouette: 0.4014


## 5. Feature engineering causal producto-período

Los ratios explosivos de v11 se reemplazan por diferencias logarítmicas y cambios simétricos,
acotados. No se usa información futura para construir las variables.


In [7]:
def construir_features(p):
    d = p.sort_values(["product_id","periodo"]).copy()
    g = d.groupby("product_id", sort=False)
    d["tn0"] = d.tn
    for k in [1,2,3,6,10,12,22]:
        d[f"lag{k}"] = g.tn.shift(k)
    for w in [3,6,12]:
        d[f"ma{w}"] = g.tn.transform(lambda s: s.shift(1).rolling(w,min_periods=1).mean())
        d[f"std{w}"] = g.tn.transform(lambda s: s.shift(1).rolling(w,min_periods=2).std())
        d[f"meses_activos_{w}"] = g.tn.transform(lambda s: s.shift(1).gt(0).rolling(w,min_periods=1).sum())
    d["trend3"] = (d.lag1-d.lag3)/2
    d["trend6"] = (d.lag1-d.lag6)/5
    d["trend12"] = (d.lag1-d.lag12)/11
    d["aceleracion"] = d.trend3-d.trend12
    d["estacional_log1y"] = (np.log1p(d.lag10.clip(lower=0))-np.log1p(d.ma12.clip(lower=0))).clip(-3,3)
    d["estacional_log2y"] = (np.log1p(d.lag22.clip(lower=0))-np.log1p(d.ma12.clip(lower=0))).clip(-3,3)
    d["frecuencia12"] = d.meses_activos_12/12
    d["adi12"] = (12/d.meses_activos_12.replace(0,np.nan)).clip(1,12).fillna(12)
    d["clientes_activos_lag1"] = g.clientes_activos.shift(1)
    d["cambio_clientes"] = ((d.clientes_activos-d.clientes_activos_lag1)/(d.clientes_activos+d.clientes_activos_lag1+1)).clip(-1,1)
    d["tasa_retencion"] = (d.clientes_retenidos/(d.clientes_activos_lag1+1)).clip(0,1)
    d["tn_por_cliente"] = d.tn/(d.clientes_activos+1)
    d["mes"] = d.periodo%100
    d["sin_mes"] = np.sin(2*np.pi*d.mes/12)
    d["cos_mes"] = np.cos(2*np.pi*d.mes/12)
    d["tn_categoria"] = d.groupby(["cat3","periodo"]).tn.transform("sum")
    d["share_categoria"] = d.tn/(d.tn_categoria+1e-6)
    idx = month_index(d.periodo)
    first = idx.where(d.tn>0).groupby(d.product_id).cummin().groupby(d.product_id).ffill()
    last = idx.where(d.tn>0).groupby(d.product_id).ffill()
    d["edad_activa"] = (idx-first).clip(lower=0).fillna(0)
    d["meses_sin_venta"] = (idx-last).clip(lower=0).fillna(36)
    d["target"] = g.tn.shift(-HORIZONTE)
    d["target_period"] = d.periodo.map(lambda x:add_months_scalar(x,HORIZONTE))
    return d

fe = construir_features(panel)
NUM_FEATURES = [
    "tn0","lag1","lag2","lag3","lag6","lag10","lag12","lag22",
    "ma3","ma6","ma12","std3","std6","std12","trend3","trend6","trend12","aceleracion",
    "estacional_log1y","estacional_log2y","meses_activos_3","meses_activos_6","meses_activos_12",
    "frecuencia12","adi12","clientes_activos","clientes_activos_lag1","cambio_clientes",
    "tasa_retencion","tn_por_cliente","hhi_clientes","share_top1","share_top5",
    "mes","sin_mes","cos_mes","tn_categoria","share_categoria","edad_activa","meses_sin_venta"
]
CAT_FEATURES = ["product_id","cat3","brand","cluster_id"]
FEATURES = NUM_FEATURES + CAT_FEATURES
for c in CAT_FEATURES:
    fe[c] = fe[c].astype("category")
print("features:", len(FEATURES), "filas:", len(fe))
print(fe[NUM_FEATURES].replace([np.inf,-np.inf],np.nan).describe().T[["min","max"]])


features: 44 filas: 44388
                              min           max
tn0                      0.000000   2295.198320
lag1                     0.000000   2295.198320
lag2                     0.000000   2295.198320
lag3                     0.000000   2295.198320
lag6                     0.000000   2295.198320
lag10                    0.000000   2295.198320
lag12                    0.000000   2295.198320
lag22                    0.000000   1958.598450
ma3                      0.000000   1864.966707
ma6                      0.000000   1717.491073
ma12                     0.000000   1575.534308
std3                     0.000000    609.142005
std6                     0.000000    524.980030
std12                    0.000000    434.456044
trend3                -498.467505    582.877100
trend6                -236.008502    267.623286
trend12                -95.114263     86.020076
aceleracion           -489.213422    563.538403
estacional_log1y        -3.000000      2.009193
estacional_log

## 6. Controles de leakage y particiones


In [8]:
assert "target" not in FEATURES and "target_period" not in FEATURES
assert fe.loc[fe.periodo==ORIGEN_INFERENCIA,"target"].isna().all()
assert add_months_scalar(ORIGEN_INFERENCIA,HORIZONTE)==202002
assert set(VALID_ORIGINS).isdisjoint(TEST_ORIGINS)
print("OK: features causales; target t+2 separado; inferencia febrero 2020")


OK: features causales; target t+2 separado; inferencia febrero 2020


## 7. Modelos multisemilla

Cada prueba promedia las tres semillas. Los especialistas se entrenan por cluster de curva.


In [9]:
PARAM_GLOBAL = dict(objective="tweedie", tweedie_variance_power=1.3, metric="mae",
    n_estimators=500, learning_rate=0.03, num_leaves=63, max_depth=8,
    min_child_samples=25, subsample=0.85, colsample_bytree=0.8,
    reg_alpha=0.05, reg_lambda=0.2, verbosity=-1, n_jobs=N_THREADS)
PARAM_CLUSTER = {**PARAM_GLOBAL, "num_leaves":31, "max_depth":6, "min_child_samples":15, "n_estimators":400}

def pesos_recencia(periodos, half_life=18):
    z = month_index(periodos).to_numpy()
    return np.power(0.5,(z.max()-z)/half_life)

def preparar(df):
    x = df[FEATURES].copy()
    x[NUM_FEATURES] = x[NUM_FEATURES].replace([np.inf,-np.inf],np.nan)
    return x

def predecir_origen(origen, final=False):
    target_p = add_months_scalar(origen,HORIZONTE)
    fit = fe[(fe.target.notna()) & (fe.target_period <= origen)].copy()
    ev = fe[fe.periodo==origen].copy()
    if len(ev)==0: raise RuntimeError(f"Sin filas para origen {origen}")
    pg=[]
    for seed in SEEDS:
        m=lgb.LGBMRegressor(**PARAM_GLOBAL, random_state=seed, bagging_seed=seed, feature_fraction_seed=seed)
        m.fit(preparar(fit),fit.target,categorical_feature=CAT_FEATURES,sample_weight=pesos_recencia(fit.periodo))
        pg.append(np.maximum(m.predict(preparar(ev)),0))
    pred_global=np.mean(pg,axis=0)

    pred_cluster=np.zeros(len(ev),dtype=float)
    for k in sorted(ev.cluster_id.astype(int).unique()):
        f=fit[fit.cluster_id.astype(int)==k]
        e=ev[ev.cluster_id.astype(int)==k]
        pos=ev.index.get_indexer(e.index)
        if len(f)<50:
            pred_cluster[pos] = pred_global[pos]
            continue
        pcs=[]
        cats_cluster=[c for c in CAT_FEATURES if c!="cluster_id"]
        feats_cluster=[c for c in FEATURES if c!="cluster_id"]
        for seed in SEEDS:
            m=lgb.LGBMRegressor(**PARAM_CLUSTER,random_state=seed,bagging_seed=seed,feature_fraction_seed=seed)
            m.fit(f[feats_cluster],f.target,categorical_feature=cats_cluster,sample_weight=pesos_recencia(f.periodo))
            pcs.append(np.maximum(m.predict(e[feats_cluster]),0))
        pred_cluster[pos]=np.mean(pcs,axis=0)
    pred_base=np.maximum(ev.lag10.fillna(ev.ma12).fillna(0).to_numpy(),0)
    out=ev[["product_id","periodo","target"]].copy()
    out["target_period"]=target_p
    out["pred_global"]=pred_global
    out["pred_cluster"]=pred_cluster
    out["pred_estacional"]=pred_base
    return out

def wape(y,p):
    den=np.asarray(y,dtype=float).sum()
    return np.abs(np.asarray(y)-np.asarray(p)).sum()/den if den>0 else np.nan


## 8. Validación multisemilla y selección de ensemble


In [10]:
pred_val=pd.concat([predecir_origen(o) for o in VALID_ORIGINS],ignore_index=True)
pred_val=pred_val[pred_val.product_id.isin(oficial.product_id)].copy()
candidatos=[]
for wc in np.arange(0,1.01,0.1):
  for wg in np.arange(0,1.01-wc,0.1):
    wb=round(1-wc-wg,10)
    raw=wc*pred_val.pred_cluster+wg*pred_val.pred_global+wb*pred_val.pred_estacional
    for cal in [0.95,0.975,1.0,1.025,1.05,1.075]:
      candidatos.append({"w_cluster":wc,"w_global":wg,"w_estacional":wb,"calibracion":cal,
                         "wape_val":wape(pred_val.target,np.maximum(raw*cal,0))})
tabla_val=pd.DataFrame(candidatos).sort_values("wape_val").reset_index(drop=True)
MEJOR=tabla_val.iloc[0].to_dict()
tabla_val.to_csv(OUT/"seleccion_ensemble_validacion.csv",index=False)
print(tabla_val.head(15).to_string(index=False))
print("MEJOR VALIDACIÓN:",MEJOR)


[LightGBM] [Warning] Met categorical feature which contains sparse values. Consider renumbering to consecutive integers started from zero
[LightGBM] [Warning] Met categorical feature which contains sparse values. Consider renumbering to consecutive integers started from zero
[LightGBM] [Warning] Met categorical feature which contains sparse values. Consider renumbering to consecutive integers started from zero
[LightGBM] [Warning] Met categorical feature which contains sparse values. Consider renumbering to consecutive integers started from zero
[LightGBM] [Warning] Met categorical feature which contains sparse values. Consider renumbering to consecutive integers started from zero
[LightGBM] [Warning] Met categorical feature which contains sparse values. Consider renumbering to consecutive integers started from zero
 w_cluster  w_global  w_estacional  calibracion  wape_val
       0.0       1.0           0.0        0.950  0.311391
       0.0       0.9           0.1        0.950  0.31170

## 9. Test temporal independiente


In [11]:
pred_test=pd.concat([predecir_origen(o) for o in TEST_ORIGINS],ignore_index=True)
pred_test=pred_test[pred_test.product_id.isin(oficial.product_id)].copy()
pred_test["pred_final"] = np.maximum(MEJOR["calibracion"]*(
    MEJOR["w_cluster"]*pred_test.pred_cluster+
    MEJOR["w_global"]*pred_test.pred_global+
    MEJOR["w_estacional"]*pred_test.pred_estacional),0)
metricas_test={
    "global":wape(pred_test.target,pred_test.pred_global),
    "cluster":wape(pred_test.target,pred_test.pred_cluster),
    "estacional":wape(pred_test.target,pred_test.pred_estacional),
    "ensemble_elegido":wape(pred_test.target,pred_test.pred_final),
}
print(pd.Series(metricas_test).sort_values())
pred_test.to_csv(OUT/"diagnostico_test_v12.csv",index=False)


[LightGBM] [Warning] Met categorical feature which contains sparse values. Consider renumbering to consecutive integers started from zero
[LightGBM] [Warning] Met categorical feature which contains sparse values. Consider renumbering to consecutive integers started from zero
[LightGBM] [Warning] Met categorical feature which contains sparse values. Consider renumbering to consecutive integers started from zero
ensemble_elegido    0.255912
global              0.273362
cluster             0.297628
estacional          0.338023
dtype: float64


## 10. Entrenamiento final y submission único


In [13]:
pred_inf=predecir_origen(ORIGEN_INFERENCIA,final=True)
pred_inf["tn"] = np.maximum(MEJOR["calibracion"]*(
    MEJOR["w_cluster"]*pred_inf.pred_cluster+
    MEJOR["w_global"]*pred_inf.pred_global+
    MEJOR["w_estacional"]*pred_inf.pred_estacional),0)
submission=oficial.merge(pred_inf[["product_id","tn"]],on="product_id",how="left",validate="one_to_one")
if submission.tn.isna().any(): raise RuntimeError("Productos sin predicción")
if submission.product_id.duplicated().any(): raise RuntimeError("Product_id duplicados")
if not np.isfinite(submission.tn).all(): raise RuntimeError("Predicciones inválidas")
PATH_SUB=OUT/"submission_v12_producto_periodo_multisemilla.csv"
submission[["product_id","tn"]].to_csv(PATH_SUB,index=True,float_format="%.10f",lineterminator="\n")
# VALIDACIÓN DETALLADA DEL CSV FÍSICO

check = pd.read_csv(PATH_SUB)

# Limpiar espacios o BOM en los encabezados
check.columns = (
    check.columns
    .astype(str)
    .str.replace("\ufeff", "", regex=False)
    .str.strip()
)

oficial_control = (
    oficial[["product_id"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

print("Columnas encontradas:", check.columns.tolist())
print("Columnas esperadas:", ["product_id", "tn"])
print("Filas CSV:", len(check))
print("Productos oficiales únicos:", len(oficial_control))
print("Duplicados:", check["product_id"].duplicated().sum())
print("Nulos:", check["tn"].isna().sum())
print("Infinitos:", (~np.isfinite(check["tn"])).sum())
print("Negativos:", (check["tn"] < 0).sum())

if check.columns.tolist() != ["product_id", "tn"]:
    raise RuntimeError(
        f"Columnas incorrectas: {check.columns.tolist()}"
    )

if len(check) != len(oficial_control):
    raise RuntimeError(
        f"Cantidad de filas incorrecta: CSV={len(check)}, "
        f"oficiales únicos={len(oficial_control)}"
    )

if check["product_id"].duplicated().any():
    duplicados = check.loc[
        check["product_id"].duplicated(False),
        "product_id"
    ].tolist()

    raise RuntimeError(
        f"Existen product_id duplicados: {duplicados[:20]}"
    )

if check["tn"].isna().any():
    raise RuntimeError("Existen predicciones nulas")

if not np.isfinite(check["tn"]).all():
    raise RuntimeError("Existen predicciones infinitas")

if (check["tn"] < 0).any():
    raise RuntimeError("Existen predicciones negativas")

# Comprobar universo y orden
faltantes = set(oficial_control.product_id) - set(check.product_id)
sobrantes = set(check.product_id) - set(oficial_control.product_id)

if faltantes or sobrantes:
    raise RuntimeError(
        f"Universo incorrecto. Faltantes={list(faltantes)[:20]}, "
        f"sobrantes={list(sobrantes)[:20]}"
    )

print("\n✅ CSV V12 VALIDADO")
print("Submission:", PATH_SUB)
print("Filas:", len(check))
print("tn total:", check["tn"].sum())
print("Pesos:", MEJOR)

[LightGBM] [Warning] Met categorical feature which contains sparse values. Consider renumbering to consecutive integers started from zero
[LightGBM] [Warning] Met categorical feature which contains sparse values. Consider renumbering to consecutive integers started from zero
[LightGBM] [Warning] Met categorical feature which contains sparse values. Consider renumbering to consecutive integers started from zero
Columnas encontradas: ['Unnamed: 0', 'product_id', 'tn']
Columnas esperadas: ['product_id', 'tn']
Filas CSV: 780
Productos oficiales únicos: 780
Duplicados: 0
Nulos: 0
Infinitos: 0
Negativos: 0


RuntimeError: Columnas incorrectas: ['Unnamed: 0', 'product_id', 'tn']

In [14]:
# REPARAR DEFINITIVAMENTE EL CSV V12 SIN REENTRENAR

check = pd.read_csv(PATH_SUB)

# Eliminar cualquier columna de índice accidental
columnas_indice = [
    c for c in check.columns
    if str(c).strip().lower().startswith("unnamed")
]

if columnas_indice:
    print("Eliminando columnas accidentales:", columnas_indice)
    check = check.drop(columns=columnas_indice)

# Conservar exclusivamente las columnas oficiales
check = check[["product_id", "tn"]].copy()

check["product_id"] = pd.to_numeric(
    check["product_id"],
    errors="raise"
).astype("int64")

check["tn"] = pd.to_numeric(
    check["tn"],
    errors="raise"
).astype("float64").clip(lower=0)

# Reconstruir con el universo y orden oficial
oficial_control = (
    oficial[["product_id"]]
    .drop_duplicates()
    .copy()
)

oficial_control["product_id"] = pd.to_numeric(
    oficial_control["product_id"],
    errors="raise"
).astype("int64")

check = oficial_control.merge(
    check,
    on="product_id",
    how="left",
    validate="one_to_one"
)

if check["tn"].isna().any():
    faltantes = check.loc[
        check["tn"].isna(),
        "product_id"
    ].tolist()

    raise RuntimeError(
        f"Productos sin predicción: {faltantes[:20]}"
    )

if not np.isfinite(check["tn"]).all():
    raise RuntimeError("Existen valores infinitos")

# Sobrescribir el archivo sin índice
check.to_csv(
    PATH_SUB,
    index=False,
    columns=["product_id", "tn"],
    float_format="%.10f",
    lineterminator="\n"
)

# Reabrir el archivo físico
verificacion = pd.read_csv(PATH_SUB)

print("Columnas finales:", verificacion.columns.tolist())
print("Filas:", len(verificacion))
print("Duplicados:", verificacion["product_id"].duplicated().sum())
print("Nulos:", verificacion["tn"].isna().sum())
print("Infinitos:", (~np.isfinite(verificacion["tn"])).sum())
print("Negativos:", (verificacion["tn"] < 0).sum())
print("TN total:", verificacion["tn"].sum())
print("Archivo:", PATH_SUB)

if verificacion.columns.tolist() != ["product_id", "tn"]:
    raise RuntimeError(
        f"El archivo continúa teniendo columnas incorrectas: "
        f"{verificacion.columns.tolist()}"
    )

if len(verificacion) != len(oficial_control):
    raise RuntimeError("Cantidad de filas incorrecta")

print("\n✅ CSV V12 REPARADO Y VALIDADO")

Eliminando columnas accidentales: ['Unnamed: 0']
Columnas finales: ['product_id', 'tn']
Filas: 780
Duplicados: 0
Nulos: 0
Infinitos: 0
Negativos: 0
TN total: 25798.137565698296
Archivo: /home/ds/buckets/b1/pipeline_v12_producto_periodo/submission_v12_producto_periodo_multisemilla.csv

✅ CSV V12 REPARADO Y VALIDADO


## 11. Envío opcional

Esta es la única celda que puede enviar a Kaggle. Primero revisar validación, test y CSV.


In [15]:
FLAG=OUT/"submit_v12.done"
if SUBMIT_V12 and not FLAG.exists():
    r=subprocess.run(["kaggle","competitions","submit","-c","labo-iii-2026-ba","-f",str(PATH_SUB),
                      "-m","v12 producto-periodo multisemilla"],capture_output=True,text=True)
    print((r.stdout or "")+(r.stderr or ""))
    if r.returncode==0: FLAG.write_text(time.strftime("%F %T"))
elif FLAG.exists(): print("La v12 ya fue enviada")
else: print("SUBMIT_V12=False: CSV generado, no enviado")


SUBMIT_V12=False: CSV generado, no enviado


## Lectura de resultados

Antes de subir, comparar:

1. mejor WAPE de validación;
2. WAPE del ensemble en test contra global, cluster y estacional;
3. pesos seleccionados: si un modelo recibe peso cero, queda descartado con evidencia;
4. total de toneladas y esquema exacto del CSV.
